# Laya V9 — Ternary CeNN (−1, 0, +1) and concat memory experiment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V9_Ternary_CeNN_Concat_Memory_Ablation_Colab.ipynb)

**Three controlled comparisons** on T4/L4: CeNN ternary stencil output `sum` versus `concat`; bit-packed G128 {-1,0,+1} projection output preallocation versus concatenation; already fused QKV versus split Q/K/V plus concat. Captures actual incremental GPU peak memory, persistent state bytes and latency.

2-bit codes store **four ternary values per byte**. One FP16 scale per 128 weights adds 0.125 bits per weight, yielding 2.125 bits/weight or ~86.7% less **projection weight storage** than FP16 before biases. No compiler builds or model downloads. **Concat is not weight compression.** This is not a native ternary GEMM or a complete trained CeNN replacement of Laya.


In [ ]:
#@title 1. Fast no-install GPU setup and exact ternary pack/unpack test
import os,sys,json,gc,zipfile,importlib.util
from pathlib import Path
from urllib.request import urlopen
import torch,numpy as np,pandas as pd
from torch import nn
from IPython.display import display
REPEATS=3 #@param {type:"integer"}
CHUNK_ROWS=64 #@param {type:"integer"}
OUT=Path("/content/Laya_V9_Ternary_CeNN_Memory")
OUT.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(),"Select a T4/L4 CUDA GPU"
CODE_SHA="4ea3bf6e5001b7b123d7c137cb09f0ce32cc1c55"
p=Path("/content/laya_cenn_ternary_concat_v9.py")
with urlopen(f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{CODE_SHA}/src/tinycenn_lm/laya_cenn_ternary_concat_v9.py",timeout=90) as r:p.write_bytes(r.read())
spec=importlib.util.spec_from_file_location("v9cenn",str(p))
v9=importlib.util.module_from_spec(spec);sys.modules[spec.name]=v9
spec.loader.exec_module(v9)
codes=torch.randint(-1,2,(8,256),device="cuda",dtype=torch.int8)
assert torch.equal(v9.unpack_ternary(v9.pack_ternary(codes),256),codes)
print("PASS: exact 2-bit ternary -1/0/+1; GPU:",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. CeNN stencil: concat versus direct signed addition
stencil=v9.TernaryCeNNStencil(
    [[-1,0,1],[1,-1,0],[0,1,-1],[1,0,0]], [1.,.5,.5,1.]).cuda()
CENN=[]
for B,T,D in ((1,128,768),(1,512,768),(4,512,768),(4,512,1024)):
    x=torch.randn(B,T,D,device="cuda",dtype=torch.float16)
    torch.testing.assert_close(stencil(x,"sum"),stencil(x,"concat"),
                               rtol=.01,atol=.05)
    s=v9.peak_cuda_forward(lambda:stencil(x,"sum"),repeats=REPEATS)
    c=v9.peak_cuda_forward(lambda:stencil(x,"concat"),repeats=REPEATS)
    CENN.append(dict(batch=B,tokens=T,features=D,
       kernel_state_bytes=v9.serialized_bytes(stencil),
       sum_peak_mib=s["peak_extra_alloc_mib"],
       concat_peak_mib=c["peak_extra_alloc_mib"],
       concat_extra_peak_mib=c["peak_extra_alloc_mib"]-s["peak_extra_alloc_mib"],
       sum_ms=s["latency_ms"],concat_ms=c["latency_ms"]))
    print("CeNN B",B,"T",T,"sum peak",round(s["peak_extra_alloc_mib"],3),
          "cat peak",round(c["peak_extra_alloc_mib"],3))
    del x
display(pd.DataFrame(CENN))
pd.DataFrame(CENN).to_csv(OUT/"cenn_concat_vs_sum.csv",index=False)


In [ ]:
#@title 3. G128 ternary weight checkpoint and tiled output cat vs preallocation
PROJECTIONS=[]
for label,K,N in (("QKV",768,2304),("MLP_up",768,3072),("MLP_down",3072,768)):
    fp16=nn.Linear(K,N,device="cuda",dtype=torch.float16).eval()
    pre=v9.PackedTernaryLinear.from_linear(fp16,CHUNK_ROWS,"prealloc")
    cat=v9.PackedTernaryLinear.from_linear(fp16,CHUNK_ROWS,"cat")
    assert not hasattr(pre,"_cached")
    for M in (128,512):
        x=torch.randn(M,K,device="cuda",dtype=torch.float16)
        a=pre(x);b=cat(x);ref=fp16(x)
        torch.testing.assert_close(a,b,rtol=.01,atol=.04)
        mse=float((a.float()-ref.float()).square().mean()/
                   ref.float().square().mean().clamp_min(1e-9))
        r=dict(layer=label,rows=M,
               fp16_weight_state=v9.serialized_bytes(fp16),
               packed_ternary_state=v9.serialized_bytes(pre),
               relative_mse=mse)
        for tag,layer in (("fp16",fp16),("prealloc",pre),("concat",cat)):
            t=v9.peak_cuda_forward(lambda l=layer:l(x),repeats=REPEATS,warmup=1)
            r[tag+"_peak_mib"]=t["peak_extra_alloc_mib"]
            r[tag+"_ms"]=t["latency_ms"]
        r["storage_saving_pct"]=100*(1-r["packed_ternary_state"]/r["fp16_weight_state"])
        PROJECTIONS.append(r)
        print(label,"M",M,"saved",round(r["storage_saving_pct"],1),"%",
              "cat vs prealloc peak difference",
              round(r["concat_peak_mib"]-r["prealloc_peak_mib"],2),"MiB")
        del x,a,b,ref
    del fp16,pre,cat
    gc.collect();torch.cuda.empty_cache()
display(pd.DataFrame(PROJECTIONS))
pd.DataFrame(PROJECTIONS).to_csv(OUT/"ternary_linear_concat_vs_prealloc.csv",index=False)


In [ ]:
#@title 4. Fused QKV versus separate Q/K/V + concat with IDENTICAL weights
D=768
fused=nn.Linear(D,3*D,device="cuda",dtype=torch.float16)
branches=[nn.Linear(D,D,device="cuda",dtype=torch.float16) for _ in range(3)]
with torch.no_grad():
    for idx,layer in enumerate(branches):
        layer.weight.copy_(fused.weight[idx*D:(idx+1)*D])
        layer.bias.copy_(fused.bias[idx*D:(idx+1)*D])
assert v9.serialized_bytes(fused)==sum(v9.serialized_bytes(x) for x in branches)
QKV=[]
for B,T in ((1,128),(1,512),(4,512)):
    x=torch.randn(B,T,D,device="cuda",dtype=torch.float16)
    def split():return torch.cat([l(x) for l in branches],dim=-1)
    torch.testing.assert_close(fused(x),split(),atol=.04,rtol=.02)
    a=v9.peak_cuda_forward(lambda:fused(x),repeats=REPEATS)
    b=v9.peak_cuda_forward(split,repeats=REPEATS)
    QKV.append(dict(batch=B,tokens=T,
      same_weight_state_bytes=v9.serialized_bytes(fused),
      fused_peak_mib=a["peak_extra_alloc_mib"],
      split_concat_peak_mib=b["peak_extra_alloc_mib"],
      extra_concat_mib=b["peak_extra_alloc_mib"]-a["peak_extra_alloc_mib"],
      fused_ms=a["latency_ms"],split_concat_ms=b["latency_ms"]))
    print("QKV B",B,"T",T,"fused",round(a["peak_extra_alloc_mib"],2),
          "MiB vs cat",round(b["peak_extra_alloc_mib"],2),"MiB")
    del x
display(pd.DataFrame(QKV))
pd.DataFrame(QKV).to_csv(OUT/"qkv_fused_vs_cat.csv",index=False)
del fused,branches
gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 5. Save real measurements + clearly marked theoretical all-28 estimate
example_shapes=[(2304,768),(768,768),(3072,768),(768,3072)]*28
estimate=v9.theoretical_pack_bytes(example_shapes)
estimate["scope"]="illustrative 112 projections, not exact Laya checkpoint size"
summary=dict(experiment="V9 strict ternary {-1,0,+1} CeNN and concat",
  gpu=torch.cuda.get_device_name(0),code_sha=CODE_SHA,
  bits_per_weight=2.125,
  stencil_state_bytes=v9.serialized_bytes(stencil),
  measured_cenn_memory=CENN,measured_ternary_projection_memory=PROJECTIONS,
  measured_qkv_memory=QKV,
  theoretical_all28_projection_storage=estimate,
  caveats=["Concat does not reduce parameter count or bit width",
           "Concat can increase GPU activation allocations",
           "Packed-state saving is distinct from peak live memory",
           "Tile reconstruction uses FP16 matmul and may be slow",
           "Only synthetic shape ablation; full-model fidelity not tested"])
(OUT/"v9_memory_summary.json").write_text(json.dumps(summary,indent=2))
zip_path=OUT/"Laya_V9_Ternary_CeNN_Concat_Memory_Results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for file in OUT.iterdir():
        if file.is_file() and file!=zip_path:archive.write(file,file.name)
print("ZIP",zip_path)
print("Estimated projection-weight reduction",round(100*estimate["saving_fraction"],2),"%")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:pass


## Interpretation

Ternary CeNN coefficients are exactly **−1, 0, +1**. 2-bit packing reduces **persistent weight storage** without a full-weight FP16 cache; G128 scales add a little overhead. Direct accumulation and already-fused QKV generally need fewer temporary activation buffers than concatenation. Compare CSV peak-memory measurements for your GPU.

This is not a trained all-28 ternary model and **not native ternary GEMM**. An accuracy-gated Laya conversion should be attempted only after inspecting these memory results.
